In [3]:
from sklearnex import patch_sklearn

patch_sklearn()

import pandas as pd
from ipynb.fs.defs.preprocessing import get_data  # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from util import summary
from imblearn.over_sampling import (
    RandomOverSampler,
    ADASYN,
    SMOTE,
    SMOTEN,
    SMOTENC,
    SVMSMOTE,
    BorderlineSMOTE,
    KMeansSMOTE,
)
from imblearn.under_sampling import (
    ClusterCentroids,
    AllKNN,
    CondensedNearestNeighbour,
    EditedNearestNeighbours,
    InstanceHardnessThreshold,
    NearMiss,
    NeighbourhoodCleaningRule,
    OneSidedSelection,
    RandomUnderSampler,
    RepeatedEditedNearestNeighbours,
    TomekLinks,
)
import optuna
from optuna import create_study, create_trial

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [ ]:
# TODO K-Fold type error regularization different models
# TODO prevent overfit on train set, on val set,...

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = get_data(
    use_val=True, test_size=0.2, val_size=0.2
)

In [5]:
y_train

25895     0
14800     0
40835     0
165981    0
189171    0
         ..
232089    0
54541     0
223541    0
217618    0
173568    0
Name: Class, Length: 170884, dtype: int64

In [6]:
# oversamplers = RandomOverSampler(random_state=42, sampling_strategy=0.003)
# undersamplers = RandomUnderSampler(random_state=42, sampling_strategy=0.01)
# threshold = 0.5

In [7]:
# X_resampled, y_resampled = oversamplers.fit_resample(X_train, y_train)
# X_resampled, y_resampled = undersamplers.fit_resample(X_resampled, y_resampled)
# clf.fit(X_resampled, y_resampled)

In [8]:
# probs = clf.predict_proba(X_test)
# y_pred = (probs >= threshold).astype(int)[:, 1]
# summary(y_test, y_pred)

In [9]:
over_samplers = {
    "RandomUnderSampler": RandomUnderSampler,
    "SMOTE": SMOTE,
    "ADASYN": ADASYN,
    "SMOTEN": SMOTEN,
    # "SMOTENC": SMOTENC,  # type error need more work
    "SVMSMOTE": SVMSMOTE,
    "BorderLineSMOTE": BorderlineSMOTE,
    "KMeansSMOTE": KMeansSMOTE,
}

In [10]:
under_samplers = {
    # "ClusterCentroids": ClusterCentroids, #too slow for hyperparameters tuning
    # "AllKNN": AllKNN, different sampling_strategy, need more work
    # "CondensedNearestNeighbour": CondensedNearestNeighbour,# ValueError: 'clean-sampling' methods do let the user specify the sampling ratio.
    # "EditedNearestNeighbours": EditedNearestNeighbours,# type error, need work
    "InstanceHardnessThreshold": InstanceHardnessThreshold,
    "NearMiss": NearMiss,
    # "NeighbourhoodCleaningRule": NeighbourhoodCleaningRule,
    # "OneSidedSelection": OneSidedSelection,# ValueError: 'clean-sampling' methods do let the user specify the sampling ratio.
    "RandomUnderSampler": RandomUnderSampler,
    # "RepeatedEditedNearestNeighbours": RepeatedEditedNearestNeighbours,
    # "TomekLinks": TomekLinks, # ValueError: 'clean-sampling' methods do let the user specify the sampling ratio.
}

In [11]:
def objective(trial: optuna.Trial):
    try:
        sampling_strategy1 = trial.suggest_float(
            "sampling_strategy1", len(y_train[y_train == 1]) / len(y_train), 1
        )
        sampling_strategy2 = trial.suggest_float(
            "sampling_strategy2", len(y_train[y_train == 1]) / len(y_train), 1
        )
        if sampling_strategy1 > sampling_strategy2:
            sampling_strategy1, sampling_strategy2 = (
                sampling_strategy2,
                sampling_strategy1,
            )
        over_sampler = over_samplers[
            trial.suggest_categorical("over_sampler", list(over_samplers.keys()))
        ](sampling_strategy=sampling_strategy1, random_state=42)
        under_sampler = under_samplers[
            trial.suggest_categorical("undersampler", list(under_samplers.keys()))
        ](sampling_strategy=sampling_strategy2)
        # cant add random_state yet because of type error, need more work
        X_resampled, y_resampled = over_sampler.fit_resample(X_train, y_train)
        X_resampled, y_resampled = under_sampler.fit_resample(X_resampled, y_resampled)

        clf = Pipeline(
            [("scaler", StandardScaler()), ("logreg", LogisticRegression(max_iter=500))]
        )
        # TODO : Add more model to fine tune like SVM, xgboost, random forest, add regularizations,.................
        clf.fit(X_resampled, y_resampled)
        threshold = trial.suggest_float("threshold", 0, 1)
        y_pred = (clf.predict_proba(X_val) >= threshold)[:, 1].astype(int)

        f1 = f1_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)
        trial.set_user_attr("f1", f1)
        trial.set_user_attr("recall", recall)
        # TODO research and change objective func so that return value is single value metric
        return (f1 + recall) / 2
    except Exception as e:
        # TODO run to get all exception possible and fix
        print(e)
        return -1

In [12]:
study = optuna.create_study(
    storage="sqlite:///optuna.db",
    study_name="study_1",
    directions=["maximize"],
    load_if_exists=True,
)

[I 2026-10-16 21:58:40,287] Using an existing study with `study_name='study_1'` instead of creating a new one.


In [22]:
study.optimize(objective, n_trials=200, n_jobs=1)

[I 2026-10-16 21:26:57,762] Trial 0 finished with value: -1.0 and parameters: {'sampling_strategy1': 0.611164271751274, 'sampling_strategy2': 0.4999564768197241, 'over_sampler': 'KMeansSMOTE', 'undersampler': 'InstanceHardnessThreshold'}. Best is trial 0 with value: -1.0.


No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.


[I 2026-10-16 21:27:24,476] Trial 1 finished with value: 0.5016098319669748 and parameters: {'sampling_strategy1': 0.535635453873955, 'sampling_strategy2': 0.7559004627168995, 'over_sampler': 'SMOTE', 'undersampler': 'InstanceHardnessThreshold', 'threshold': 0.048631382966579983}. Best is trial 1 with value: 0.5016098319669748.
[I 2026-10-16 21:27:51,821] Trial 2 finished with value: 0.6188637617209045 and parameters: {'sampling_strategy1': 0.9671598243513111, 'sampling_strategy2': 0.8429121432449548, 'over_sampler': 'SVMSMOTE', 'undersampler': 'InstanceHardnessThreshold', 'threshold': 0.21423568065181842}. Best is trial 2 with value: 0.6188637617209045.
[I 2026-10-16 21:27:53,652] Trial 3 finished with value: -1.0 and parameters: {'sampling_strategy1': 0.16304653345468656, 'sampling_strategy2': 0.29488906925627745, 'over_sampler': 'KMeansSMOTE', 'undersampler': 'RandomUnderSampler'}. Best is trial 2 with value: 0.6188637617209045.


No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.


[I 2026-10-16 21:27:56,038] Trial 4 finished with value: -1.0 and parameters: {'sampling_strategy1': 0.5901533573573391, 'sampling_strategy2': 0.28657535459132316, 'over_sampler': 'KMeansSMOTE', 'undersampler': 'InstanceHardnessThreshold'}. Best is trial 2 with value: 0.6188637617209045.


...

[I 2026-10-16 21:31:09,941] Trial 49 finished with value: 0.8383275261324041 and parameters: {'sampling_strategy1': 0.9011786520441323, 'sampling_strategy2': 0.6665303598964403, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.9499033103953827}. Best is trial 44 with value: 0.8444358875625722.
[I 2026-10-16 21:31:11,149] Trial 50 finished with value: 0.6080608714073865 and parameters: {'sampling_strategy1': 0.46822363584680127, 'sampling_strategy2': 0.3268237595599952, 'over_sampler': 'BorderLineSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.13234519628053065}. Best is trial 44 with value: 0.8444358875625722.
[I 2026-10-16 21:31:16,226] Trial 51 finished with value: 0.8522370486656201 and parameters: {'sampling_strategy1': 0.9431434796203123, 'sampling_strategy2': 0.6160280249959956, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.9453701361831}. Best is trial 51 with value: 0.8522370486656201.
[I 2026-10-16

No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.


[I 2026-10-16 21:40:26,258] Trial 126 finished with value: 0.6793460839833315 and parameters: {'sampling_strategy1': 0.7204483152225691, 'sampling_strategy2': 0.8712334569818355, 'over_sampler': 'SVMSMOTE', 'undersampler': 'InstanceHardnessThreshold', 'threshold': 0.3464042475952967}. Best is trial 66 with value: 0.8586734693877551.
[I 2026-10-16 21:40:31,105] Trial 127 finished with value: 0.8562512383594214 and parameters: {'sampling_strategy1': 0.577311939143912, 'sampling_strategy2': 0.9139643263908499, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.9338594420117167}. Best is trial 66 with value: 0.8586734693877551.
[I 2026-10-16 21:40:35,790] Trial 128 finished with value: 0.8528138528138528 and parameters: {'sampling_strategy1': 0.6465712747099034, 'sampling_strategy2': 0.8276117448840911, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.988784269520059}. Best is trial 66 with value: 0.8586734693877551.
[I 2026-10-

No clusters found with sufficient samples of class 1. Try lowering the cluster_balance_threshold or increasing the number of clusters.


[I 2026-10-16 21:41:45,534] Trial 144 finished with value: 0.8565590415270585 and parameters: {'sampling_strategy1': 0.6899223453068632, 'sampling_strategy2': 0.7404898230334862, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.9673684491747949}. Best is trial 66 with value: 0.8586734693877551.
[I 2026-10-16 21:41:50,852] Trial 145 finished with value: 0.8485714285714285 and parameters: {'sampling_strategy1': 0.6714341109234939, 'sampling_strategy2': 0.808398538091329, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.969479276949914}. Best is trial 66 with value: 0.8586734693877551.
[I 2026-10-16 21:41:55,833] Trial 146 finished with value: 0.8476867186665242 and parameters: {'sampling_strategy1': 0.6504398780878016, 'sampling_strategy2': 0.652074512054926, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.99770471742839}. Best is trial 66 with value: 0.8586734693877551.
[I 2026-10-16 21:41:5

In [66]:
def test(params):
    print(params)
    sampling_strategy1 = params["sampling_strategy1"]
    sampling_strategy2 = params["sampling_strategy2"]
    if sampling_strategy1 > sampling_strategy2:
        sampling_strategy1, sampling_strategy2 = (
            sampling_strategy2,
            sampling_strategy1,
        )
    over_sampler = over_samplers[params["over_sampler"]](
        sampling_strategy=sampling_strategy1, random_state=42
    )
    under_sampler = under_samplers[params["undersampler"]](
        sampling_strategy=sampling_strategy2
    )
    # cant add random_state yet because of type error, need more work
    X_resampled, y_resampled = over_sampler.fit_resample(X_train, y_train)
    X_resampled, y_resampled = under_sampler.fit_resample(X_resampled, y_resampled)

    clf = Pipeline(
        [("scaler", StandardScaler()), ("logreg", LogisticRegression(max_iter=500))]
    )
    # TODO : Add more model to fine tune like SVM, xgboost, random forest, add regularizations,.................
    clf.fit(X_resampled, y_resampled)
    threshold = params["threshold"]
    y_pred = (clf.predict_proba(X_test) >= threshold)[:, 1].astype(int)
    summary(y_test, y_pred)

In [70]:
test(
    sorted(
        [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE],
        key=lambda t: t.value,
        reverse=(study.direction == optuna.study.StudyDirection.MAXIMIZE),
    )[1].params
)

{'sampling_strategy1': 0.5997087676064441, 'sampling_strategy2': 0.8987696627821626, 'over_sampler': 'SVMSMOTE', 'undersampler': 'RandomUnderSampler', 'threshold': 0.9674097184716919}
accuracy_score 0.9992977774656788
recall_score: 0.7864077669902912
precision_score: 0.8181818181818182
f1_score 0.801980198019802
COnfusion matrix: [[56841    18]
 [   22    81]]
